# Quantum Power Flow reproduction
This notebook provides a compact interactive path through the repository. Run it after `pip install -e .`.


In [ ]:
import numpy as np
from qpf_repro.cases import paper_case5, stressed_paper_case5
from qpf_repro.network import fast_decoupled_matrices
from qpf_repro.powerflow import solve_fast_decoupled, solve_newton_raphson
from qpf_repro.quantum import QiskitHHLFactory


## Test I: classical and quantum fast-decoupled power flow


In [ ]:
case = paper_case5()
classical = solve_fast_decoupled(case, fixed_iterations=6)
factory = QiskitHHLFactory()
quantum = solve_fast_decoupled(case, linear_system_factory=factory, fixed_iterations=6)
np.column_stack((quantum.voltage_magnitude, quantum.voltage_angle_rad))


In [ ]:
max(abs(quantum.voltage_magnitude - classical.voltage_magnitude)), quantum.max_mismatch


## Inspect the HHL spectrum and circuit statistics


In [ ]:
B_prime, _ = fast_decoupled_matrices(case)
prepared = factory.prepare(B_prime, label='B_prime')
prepared.calibration.to_dict(), prepared.circuit_statistics()


## Test II: stressed bus-1 load


In [ ]:
stressed = solve_fast_decoupled(
    stressed_paper_case5(),
    linear_system_factory=factory,
    tolerance=1e-5,
)
stressed.iterations, stressed.voltage_magnitude


## Generate the complete result bundle


In [ ]:
from qpf_repro.experiments import reproduce_all
# metrics = reproduce_all('results')
